<a href="https://colab.research.google.com/github/linasihombing/Pemrograman-Dasar/blob/main/Tugas%209.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# SEL 1: DATA
# --- Data Unit 1, 1-30 November 2026 (hari ke-1 di indeks 0) ---
energi = [5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
          6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
          5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062]      # MWh
batubara = [2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
            3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
            3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240]    # t

jam_operasi = [24] * 30
jam_operasi[16] = 18          # hari ke-17 hanya 18 jam

assert len(energi) == len(batubara) == len(jam_operasi) == 30



# SEL 2: LIMA FUNGSI + UJI (BUTIR 1 DAN 5)
def heat_rate(batubara_t, energi_mwh, nilai_kalor_kcal_kg=4400):
    """Heat rate (kcal/kWh) dari batubara (t) dan energi terkirim (MWh)."""
    if energi_mwh <= 0:
        raise ValueError("energi_mwh harus positif")
    return batubara_t * nilai_kalor_kcal_kg / energi_mwh


def efisiensi_termal(heat_rate_kcal_kwh):
    """Efisiensi termal (%) dari heat rate. 1 kWh = 860 kcal."""
    return 860 / heat_rate_kcal_kwh * 100


def capacity_factor(energi_mwh, daya_mw=330, jam=24):
    """Capacity factor (%) = energi / (daya terpasang x jam) x 100."""
    return energi_mwh / (daya_mw * jam) * 100


def konsumsi_spesifik(batubara_t, energi_mwh):
    """Konsumsi batubara spesifik (t/MWh)."""
    return batubara_t / energi_mwh


def status_hari(efisiensi, jam_operasi):
    """'TIDAK PENUH' jika jam < 24, 'EFISIENSI RENDAH' jika efisiensi < 36,3 %, selain itu 'NORMAL'."""
    if jam_operasi < 24:
        return "TIDAK PENUH"
    if efisiensi < 36.3:
        return "EFISIENSI RENDAH"
    return "NORMAL"


# Uji dengan hitungan manual
assert heat_rate(1, 2) == 2200                      # 1 x 4400 / 2
assert heat_rate(1, 2, nilai_kalor_kcal_kg=5100) == 2550   # 1 x 5100 / 2
assert round(efisiensi_termal(2150), 2) == 40       # 860 / 2150 = 0,4
assert round(capacity_factor(7920)) == 100          # 330 x 24 = 7920 MWh
assert round(capacity_factor(7200, daya_mw=600)) == 50     # 600 x 24 = 14400 -> 7200 = 50 %
assert konsumsi_spesifik(100, 200) == 0.5           # 100 / 200

# batas status: tepat di 36,3 dan prioritas "TIDAK PENUH"
assert status_hari(36.29, 24) == "EFISIENSI RENDAH"
assert status_hari(36.3, 24) == "NORMAL"
assert status_hari(40.0, 18) == "TIDAK PENUH"
assert status_hari(30.0, 18) == "TIDAK PENUH"       # jam < 24 dicek lebih dulu
print("Uji lima fungsi lulus")



# SEL 3: PERHITUNGAN HARIAN + PROSEDUR LAPORAN (BUTIR 2 DAN 3)
def hitung_harian(daya_mw, nilai_kalor, energi, batubara, jam_operasi):
    """Kembalikan lima list harian: (heat_rate, efisiensi, cf, konsumsi_spesifik, status)."""
    hr_list, efi_list, cf_list, scc_list, status_list = [], [], [], [], []
    for i in range(len(energi)):
        hr = heat_rate(batubara[i], energi[i], nilai_kalor_kcal_kg=nilai_kalor)
        efi = efisiensi_termal(hr)
        hr_list.append(hr)
        efi_list.append(efi)
        cf_list.append(capacity_factor(energi[i], daya_mw=daya_mw))
        scc_list.append(konsumsi_spesifik(batubara[i], energi[i]))
        status_list.append(status_hari(efi, jam_operasi[i]))
    return hr_list, efi_list, cf_list, scc_list, status_list


def cetak_laporan(hr_list, efi_list, cf_list, scc_list, status_list):
    """Prosedur: mencetak tabel KPI harian (satu baris per hari)."""
    print(f"{'Hari':>4}  {'HR kcal':>7}  {'Efi %':>6}  {'CF %':>5}  {'SCC':>6}  Status")
    for i in range(len(hr_list)):
        print(f"{i + 1:>4}  {hr_list[i]:>7,.0f}  {efi_list[i]:>6.2f}  "
              f"{cf_list[i]:>5.1f}  {scc_list[i]:>6.3f}  {status_list[i]}")


def cetak_ringkasan(daya_mw, energi, efi_list, status_list):
    """Prosedur: rata-rata efisiensi, CF periode, jumlah hari per status, tiga hari terendah."""
    print(f"\nRata-rata efisiensi : {sum(efi_list) / len(efi_list):.2f} %")
    cf_periode = capacity_factor(sum(energi), daya_mw=daya_mw, jam=24 * len(energi))
    print(f"Capacity factor     : {cf_periode:.1f} %")

    print("Jumlah hari per status:")
    for status in ["NORMAL", "EFISIENSI RENDAH", "TIDAK PENUH"]:
        print(f"  {status:<17}: {status_list.count(status)} hari")

    pasangan = []
    for i in range(len(efi_list)):
        pasangan.append((efi_list[i], i + 1))     # (efisiensi, nomor hari)
    pasangan.sort()                               # terkecil di depan
    print("Tiga hari paling tidak efisien:")
    for efi, hari in pasangan[:3]:
        print(f"  Hari {hari}: {efi:.2f} %")


def laporan_unit(nama, daya_mw, nilai_kalor, energi, batubara, jam_operasi):
    """Prosedur: laporan lengkap satu unit (tabel + ringkasan)."""
    print(f"=== {nama} ({daya_mw} MW, {nilai_kalor} kcal/kg) ===")
    hr, efi, cf, scc, status = hitung_harian(daya_mw, nilai_kalor, energi, batubara, jam_operasi)
    cetak_laporan(hr, efi, cf, scc, status)
    cetak_ringkasan(daya_mw, energi, efi, status)


# Uji hitung_harian: 1 hari, energi 2 MWh, batubara 1 t, 24 jam
hr, efi, cf, scc, status = hitung_harian(330, 4400, [2], [1], [24])
assert hr[0] == 2200 and round(efi[0], 2) == 39.09
assert scc[0] == 0.5 and status[0] == "NORMAL"

# Uji prosedur: tidak mengembalikan nilai (None)
assert cetak_laporan(hr, efi, cf, scc, status) is None
assert cetak_ringkasan(330, [2], efi, status) is None
print("Uji hitung_harian dan prosedur lulus")



# SEL 4: LAPORAN UNIT 1 (30 HARI)
laporan_unit("Unit 1", 330, 4400, energi, batubara, jam_operasi)



# SEL 5: PAKAI ULANG UNTUK UNIT 2 (BUTIR 4)
energi_u2 = [12150, 12080, 11020, 12210, 12095]
batubara_u2 = [5240, 5225, 4790, 5260, 5230]
jam_u2 = [24] * 5

laporan_unit("Unit 2", 600, 5100, energi_u2, batubara_u2, jam_u2)

Uji lima fungsi lulus
Hari  HR kcal   Efi %   CF %     SCC  Status
   1    2,200   39.09    0.0   0.500  NORMAL

Rata-rata efisiensi : 39.09 %
Capacity factor     : 0.0 %
Jumlah hari per status:
  NORMAL           : 1 hari
  EFISIENSI RENDAH : 0 hari
  TIDAK PENUH      : 0 hari
Tiga hari paling tidak efisien:
  Hari 1: 39.09 %
Uji hitung_harian dan prosedur lulus
=== Unit 1 (330 MW, 4400 kcal/kg) ===
Hari  HR kcal   Efi %   CF %     SCC  Status
   1    2,366   36.35   70.4   0.538  NORMAL
   2    2,351   36.58   77.0   0.534  NORMAL
   3    2,353   36.55   76.7   0.535  NORMAL
   4    2,356   36.51   77.0   0.535  NORMAL
   5    2,354   36.54   76.7   0.535  NORMAL
   6    2,350   36.59   76.7   0.534  NORMAL
   7    2,367   36.34   70.3   0.538  NORMAL
   8    2,364   36.38   70.1   0.537  NORMAL
   9    2,350   36.60   77.0   0.534  NORMAL
  10    2,357   36.49   76.8   0.536  NORMAL
  11    2,348   36.63   77.0   0.534  NORMAL
  12    2,348   36.63   76.9   0.534  NORMAL
  13    2,3